# 10 — Analisis pasca-hasil (B1–B5)

**STATUS: PASCA-HASIL.** Semua analisis di sini ditambahkan **setelah** hasil uji 2020 dan XAI utama dilihat. Aturannya:

1. **Tidak ada pemilihan model.** Model utama (seed 42, `models/`) dan semua angka di 08/09 **tidak berubah**. Notebook ini tidak menimpa `models/` maupun `predictions.parquet`.
2. **Semua varian dilaporkan**, tidak ada yang dipilih karena hasilnya bagus.
3. Di laporan, diberi label *analisis tambahan / kekokohan*. Kesimpulan utama tetap dari konfigurasi yang ditetapkan sebelumnya.

| Bagian | Pertanyaan | Keluaran |
|---|---|---|
| B1 | Apakah selisih RMSE antar model lebih besar dari variasi akibat **seed**? (10 seed, hyperparameter tetap) | `post_seed_*.csv`, `figures/post_seed_rmse.png` |
| B2 | Apakah peringkat kelompok bergantung pada **titik pembanding Shapley**? (rata-rata vs distribusi 50 hari latih) | `post_shapley_baseline*.csv` |
| B3 | Apakah peringkat kelompok **stabil antar tahun**? (XAI pada validasi 2019 vs uji 2020) | `post_xai_val2019*.csv` |
| B4 | Metode XAI mana yang paling **setia (faithful)** ke model? (uji deletion/insertion tingkat kelompok) | `post_faithfulness*.csv`, `figures/post_faithfulness.png` |
| B5 | Dari mana **bias** model berasal? (MBE per jam/musim/kondisi langit + dekomposisi galat) | `post_bias_*.csv`, `figures/post_bias_by_hour.png` |

Syarat: 07–09 sudah dijalankan (memakai `models/`, `predictions.parquet`, dan `results/xai_*.csv`). `QUICK = True` hanya untuk uji cepat (jumlah seed & latar dikurangi), **jangan** dipakai untuk angka laporan.

## 0. Setup

In [ ]:
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import importlib, evaluate as E, models as M, xai as X, lstm as L, clearsky, report
for mod in (clearsky, E, M, X, L, report): importlib.reload(mod)
from utils import set_seed

QUICK = False                                    # True HANYA untuk uji cepat
N_SEEDS, K_BG, N_RAND = (3, 8, 20) if QUICK else (10, 50, 200)

CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
PROC, RES, FIG, MOD = ROOT / "data/processed", ROOT / "results", ROOT / "figures", ROOT / "models"
SEED = set_seed(CFG["seed"]); KZ = CFG["physics"]["k_max_by_zenith"]
FMAP, APRIORI = X.load_apriori(ROOT / "configs" / "apriori_order.yaml")
FEATS = json.loads((RES / "feature_freeze.json").read_text())["fitur"]
BEST = json.loads((MOD / "best_params.json").read_text())
GROUPS = [g for g in APRIORI.index if any(FMAP.get(f) == g for f in FEATS)]
APRIORI = APRIORI[GROUPS]; MG = len(GROUPS)
MODELS = ["xgb", "rf", "lstm"]

df = pd.read_parquet(PROC / "features_all.parquet")
hourly = pd.read_parquet(ROOT / "data/interim/nsrdb_hourly_kc.parquet")
P = pd.read_parquet(PROC / "predictions.parquet")
TRAIN = (df["split"] == "train") & df["is_day"]
cs = df["ghi_cs"].fillna(0).to_numpy()
BASE = ["pred_clim", "pred_persist", "pred_gfs_raw", "pred_mos"]; ML = ["pred_rf", "pred_xgb", "pred_lstm"]
# baris evaluasi = sama persis dengan 08 (mask valid & semua prediksi utama tersedia)
EVAL = {s: (E.masks(P)["valid"] & (P["split"] == s) & P[BASE + ML].notna().all(axis=1)).to_numpy() for s in ("val", "test")}
Y = P["y_ghi"].to_numpy()

TAB = {k: M.load_model(k, MOD / f"{k}_best") for k in ("xgb", "rf")}
A = L.build_arrays(hourly, df)
NORM = L.Normalizer.load(MOD / "lstm_normalizer.npz")
XS, XT = NORM.transform(A)
NET = L.load_net(MOD / "lstm_best.pt", BEST["lstm"], XS.shape[2], XT.shape[2])
rows = A["row_index"]
POS = {r: (d, h) for d in range(rows.shape[0]) for h, r in enumerate(rows[d])}
seq_g = [L.SEQ_GROUPS[c] for c in A["seq_ch"]]; tgt_g = [L.TGT_GROUPS[c] for c in A["tgt_ch"]]
TRD, VAD = (np.where(A["split"] == s)[0] for s in ("train", "val"))

def kc_rows(kc_days):
    return pd.Series(kc_days.ravel(), index=rows.ravel()).reindex(df.index).to_numpy()

def rmse(pred, split):
    m = EVAL[split]; return float(np.sqrt(np.mean((pred[m] - Y[m]) ** 2)))

print(f"QUICK={QUICK} | seed: {N_SEEDS} | latar: {K_BG} hari | baris eval val {EVAL['val'].sum()}, uji {EVAL['test'].sum()}")
for k in MODELS:
    print(f"  {k}: RMSE uji utama {rmse(P[f'pred_{k}'].to_numpy(), 'test'):.2f}")

## B1. Variasi seed — 10 seed per model, hyperparameter tetap (`best_params.json`)

Model utama memakai seed 42. Di sini setiap model dilatih ulang dengan 10 seed (termasuk 42 sebagai cek reproduksi). **Prediksi seed lain tidak menggantikan model utama**; hanya dipakai untuk mengukur sebaran.

In [ ]:
SEEDS = [SEED] + [s for s in range(1, 1000) if s != SEED][:N_SEEDS - 1]
rec, SEEDP = [], {}
t0 = time.time()
for s in SEEDS:
    for kind in ("rf", "xgb"):
        m = M.fit_model(kind, BEST[kind], df, FEATS, TRAIN, "kc", s)
        SEEDP[(kind, s)] = M.finalize(m.predict(df[FEATS]) * cs, df, KZ, do_clip=True)
    net, hist = L.train(BEST["lstm"], XS, XT, A["y"], A["w_train"], A["w_eval"], TRD, VAD, s)
    SEEDP[("lstm", s)] = M.finalize(kc_rows(L.predict(net, XS, XT)) * cs, df, KZ, do_clip=True)
    for kind in MODELS:
        p = SEEDP[(kind, s)]
        rec.append({"model": kind, "seed": s, "rmse_val": rmse(p, "val"), "rmse_uji": rmse(p, "test"),
                    "mbe_uji": float(np.mean(p[EVAL["test"]] - Y[EVAL["test"]])),
                    "epoch_lstm": len(hist) if kind == "lstm" else np.nan})
    print(f"seed {s:3d} selesai ({time.time() - t0:.0f} dtk)")
SR = pd.DataFrame(rec)
clim_uji = rmse(P["pred_clim"].to_numpy(), "test")
SR["skill_uji_vs_clim"] = 1 - SR["rmse_uji"] / clim_uji
SR.round(4).to_csv(RES / "post_seed_rmse.csv", index=False)
pd.DataFrame({f"{k}_s{s}": v for (k, s), v in SEEDP.items()}, index=df.index).to_parquet(PROC / "post_seed_predictions.parquet")

# cek reproduksi: seed 42 harus sama dengan model utama
for k in MODELS:
    d = np.nanmax(np.abs(SEEDP[(k, SEED)] - P[f"pred_{k}"].to_numpy()))
    print(f"{k}: seed {SEED} vs model utama, selisih maks {d:.4f} W/m²", "✓" if d < 1e-3 else "⚠ tidak identik — periksa")

In [ ]:
summ = {}
for k in MODELS:
    g = SR[SR["model"] == k]
    for col, name in (("rmse_val", "RMSE val 2019"), ("rmse_uji", "RMSE uji 2020"), ("skill_uji_vs_clim", "skill uji vs klim")):
        v = g[col]
        summ[(k, name)] = {"rata2": v.mean(), "std": v.std(ddof=1), "min": v.min(), "maks": v.max(),
                           f"seed{SEED}": float(g.loc[g["seed"] == SEED, col].iloc[0])}
summ = pd.DataFrame(summ).T
summ.index.names = ["model", "besaran"]
display(summ.round(3)); summ.round(4).to_csv(RES / "post_seed_summary.csv")

# selisih antar model: semua pasangan seed (a_i − b_j)
pairs = []
for a, b in (("lstm", "rf"), ("lstm", "xgb"), ("xgb", "rf")):
    ra = SR.loc[SR["model"] == a, "rmse_uji"].to_numpy(); rb = SR.loc[SR["model"] == b, "rmse_uji"].to_numpy()
    d = (ra[:, None] - rb[None, :]).ravel()
    main = float(SR.query("model == @a and seed == @SEED")["rmse_uji"].iloc[0] - SR.query("model == @b and seed == @SEED")["rmse_uji"].iloc[0])
    pairs.append({"pasangan": f"{a} − {b}", f"selisih_seed{SEED}": main, "rata2_selisih": d.mean(),
                  "min": d.min(), "maks": d.max(), "%_pasangan_a_lebih_buruk": 100 * (d > 0).mean(),
                  "std_seed_a": ra.std(ddof=1), "std_seed_b": rb.std(ddof=1)})
pairs = pd.DataFrame(pairs).round(3)
display(pairs); pairs.to_csv(RES / "post_seed_pairs.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
for a, col, title in zip(axs, ("rmse_val", "rmse_uji"), ("Validasi 2019", "Uji 2020")):
    for i, k in enumerate(MODELS):
        g = SR[SR["model"] == k]
        xj = np.full(len(g), i) + np.linspace(-.12, .12, len(g))
        a.scatter(xj, g[col], s=18, alpha=.7)
        main = (g["seed"] == SEED).to_numpy()
        a.scatter(xj[main], g.loc[main, col], s=90, facecolors="none", edgecolors="k", label="seed 42 (utama)" if i == 0 else None)
    a.set_xticks(range(len(MODELS))); a.set_xticklabels([k.upper() for k in MODELS])
    a.set(title=f"RMSE per seed — {title}", ylabel="RMSE (W/m²)")
axs[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "post_seed_rmse.png", dpi=150, bbox_inches="tight"); plt.show()

## B2. Sensitivitas titik pembanding Shapley kelompok

Di 09, kelompok yang "dihapus" diganti **rata-rata data latih** (satu titik). Di sini diganti nilai dari **50 hari latih acak** lalu dirata-rata (Shapley intervensional):
- **Model pohon:** baris pada **jam yang sama** dari hari-hari latar.
- **LSTM:** sekuens dan blok hari latar utuh, sehingga kanal jam tetap selaras.

Sampel yang dipakai sama dengan 09 (`xai_sample_idx.csv`). Dugaan yang diuji: peringkat **geometri_waktu** LSTM (#1 dengan rata-rata, #3 dengan GradientExplainer) adalah efek titik pembanding.

In [ ]:
IDX = pd.read_csv(RES / "xai_sample_idx.csv")["row"].to_numpy()
DPOS = np.array([POS[r][0] for r in IDX]); HPOS = np.array([POS[r][1] for r in IDX])
Xs = df.loc[IDX, FEATS]
rng = np.random.default_rng(SEED)
BG_DAYS = np.sort(rng.choice(TRD, K_BG, replace=False))
bg_rows = rows[BG_DAYS][:, HPOS].T                                     # [sampel, K]: jam sama, hari latar
BG_TAB = df.loc[bg_rows.ravel(), FEATS].to_numpy(float).reshape(len(IDX), K_BG, len(FEATS))
bgmean = df.loc[TRAIN, FEATS].mean()

VF = {("xgb", "rata-rata"): X.tabular_value_fn(TAB["xgb"], Xs, bgmean, FMAP, GROUPS),
      ("rf", "rata-rata"): X.tabular_value_fn(TAB["rf"], Xs, bgmean, FMAP, GROUPS),
      ("lstm", "rata-rata"): L.lstm_value_fn(NET, XS[DPOS], XT[DPOS], HPOS, seq_g, tgt_g, GROUPS),
      ("xgb", "distribusi"): X.tabular_value_fn_bg(TAB["xgb"], Xs, BG_TAB, FMAP, GROUPS),
      ("rf", "distribusi"): X.tabular_value_fn_bg(TAB["rf"], Xs, BG_TAB, FMAP, GROUPS),
      ("lstm", "distribusi"): L.lstm_value_fn_bg(NET, XS[DPOS], XT[DPOS], HPOS, seq_g, tgt_g, GROUPS,
                                                 XS[BG_DAYS], XT[BG_DAYS])}
PHI, V = {}, {}
for key, fn in VF.items():
    t0 = time.time()
    phi, v = X.exact_group_shapley(fn, GROUPS, return_values=True)
    PHI[key], V[key] = phi.set_axis(IDX), v
    print(f"{key}: {time.time() - t0:.0f} dtk")
for k in MODELS:                                                       # cek: sama dengan 09
    old = pd.read_csv(RES / f"xai_group_shapley_{k}.csv", index_col=0)[GROUPS].reindex(IDX).to_numpy()
    d = np.abs(old - PHI[(k, "rata-rata")][GROUPS].to_numpy()).max()
    print(f"{k}: Shapley rata-rata vs 09, selisih maks {d:.2e}", "✓" if d < 1e-6 else "⚠")

In [ ]:
por = pd.DataFrame({f"{k}|{b}": PHI[(k, b)].abs().mean() / PHI[(k, b)].abs().mean().sum()
                    for k in MODELS for b in ("rata-rata", "distribusi")}).loc[GROUPS]
ge_path = RES / "xai_ge_lstm_groups.csv"
if ge_path.exists():
    ge = pd.read_csv(ge_path, header=[0, 1], index_col=0)
    por["lstm|GradientExplainer"] = ge[("porsi", "GE |Σφ| kelompok")].reindex(GROUPS)
rank = por.rank(ascending=False).astype(int)
tab_b2 = pd.concat({"porsi": por.round(3), "peringkat": rank}, axis=1); tab_b2.insert(0, "a_priori", APRIORI)
display(tab_b2); tab_b2.to_csv(RES / "post_shapley_baseline.csv"); rank.to_csv(RES / "post_shapley_baseline_rank.csv")

agr = []
for k in MODELS:
    a, b = PHI[(k, "rata-rata")].abs(), PHI[(k, "distribusi")].abs()
    agr.append({"model": k, "pembanding": "rata-rata vs distribusi", "rho": spearmanr(a.mean(), b.mean()).statistic,
                "jaccard_top3": X.jaccard_topk(a.mean(), b.mean(), 3)})
    for base in ("rata-rata", "distribusi"):
        agr.append({"model": k, "pembanding": f"{base} vs a priori",
                    **X.spearman_vs_apriori(PHI[(k, base)].abs(), APRIORI, n_boot=1000, seed=SEED)})
if "lstm|GradientExplainer" in por:
    for base in ("rata-rata", "distribusi"):
        agr.append({"model": "lstm", "pembanding": f"{base} vs GradientExplainer",
                    "rho": spearmanr(por[f"lstm|{base}"], por["lstm|GradientExplainer"]).statistic,
                    "jaccard_top3": X.jaccard_topk(por[f"lstm|{base}"], por["lstm|GradientExplainer"], 3)})
agr = pd.DataFrame(agr).round(3)
display(agr); agr.to_csv(RES / "post_shapley_baseline_agreement.csv", index=False)
print("Peringkat geometri_waktu LSTM:", {c: int(rank.loc["geometri_waktu", c]) for c in rank if c.startswith("lstm")})
print("Peringkat nwp_radiasi:", {c: int(rank.loc["nwp_radiasi", c]) for c in rank})

## B3. Stabilitas antar tahun — XAI pada validasi 2019

Prosedur sama dengan 09 (200 jam siang-valid terstratifikasi musim × kondisi langit; metrik utama: TreeSHAP tingkat fitur untuk pohon, Shapley kelompok eksak untuk ketiga model). Hasil 2020 dibaca dari `results/` (09).

In [ ]:
val = df[(df["split"] == "val") & E.masks(df)["valid"]]
IDXV = X.stratified_sample(val, 200, SEED)
DPOSV = np.array([POS[r][0] for r in IDXV]); HPOSV = np.array([POS[r][1] for r in IDXV])
XsV = df.loc[IDXV, FEATS]
G19, G20 = {}, {}
sh19 = {"xgb": X.xgb_shap(TAB["xgb"], XsV), "rf": X.rf_shap(TAB["rf"], XsV)}
for k in ("xgb", "rf"):
    G19[(k, "TreeSHAP fitur")] = X.group_abs(sh19[k], FMAP)[GROUPS]
    G20[(k, "TreeSHAP fitur")] = X.group_abs(pd.read_csv(RES / f"xai_shap_{k}.csv", index_col=0), FMAP)[GROUPS]
vf19 = {k: X.tabular_value_fn(TAB[k], XsV, bgmean, FMAP, GROUPS) for k in ("xgb", "rf")}
vf19["lstm"] = L.lstm_value_fn(NET, XS[DPOSV], XT[DPOSV], HPOSV, seq_g, tgt_g, GROUPS)
for k in MODELS:
    G19[(k, "Shapley kelompok")] = X.exact_group_shapley(vf19[k], GROUPS).set_axis(IDXV).abs()
    G20[(k, "Shapley kelompok")] = pd.read_csv(RES / f"xai_group_shapley_{k}.csv", index_col=0)[GROUPS].abs()

por3, agr3 = {}, []
for key in G19:
    p19 = G19[key].mean() / G19[key].mean().sum(); p20 = G20[key].mean() / G20[key].mean().sum()
    por3[f"{key[0]}|{key[1]}|2019"], por3[f"{key[0]}|{key[1]}|2020"] = p19, p20
    r19 = X.spearman_vs_apriori(G19[key], APRIORI, n_boot=1000, seed=SEED)
    agr3.append({"model": key[0], "metode": key[1], "rho_2019_vs_2020": spearmanr(p19, p20).statistic,
                 "jaccard_top3_2019_vs_2020": X.jaccard_topk(p19, p20, 3),
                 "rho_apriori_2019": r19["rho"], "rho_apriori_2020": X.spearman_vs_apriori(G20[key], APRIORI, n_boot=50, seed=SEED)["rho"]})
por3 = pd.DataFrame(por3).loc[GROUPS]
tab_b3 = pd.concat({"porsi": por3.round(3), "peringkat": por3.rank(ascending=False).astype(int)}, axis=1)
tab_b3.insert(0, "a_priori", APRIORI)
display(tab_b3); tab_b3.to_csv(RES / "post_xai_val2019.csv")
agr3 = pd.DataFrame(agr3).round(3); display(agr3); agr3.to_csv(RES / "post_xai_val2019_agreement.csv", index=False)

## B4. Faithfulness — uji deletion & insertion tingkat kelompok

Untuk setiap sampel uji (sama dengan 09), kelompok diurutkan menurut |atribusi| dari tiap metode.
- **Deletion:** kelompok terpenting dihapus lebih dulu. Metode yang setia membuat prediksi **cepat berubah**, jadi AUC deletion **besar** lebih baik.
- **Insertion:** mulai dari semua dihapus, lalu kelompok terpenting dimasukkan lebih dulu. Prediksi harus **cepat kembali**, jadi AUC insertion **kecil** lebih baik.
- **Pembanding:** urutan acak (200 permutasi).

"Menghapus" dilakukan dengan dua operator, yaitu nilai rata-rata dan distribusi 50 hari latih, memakai nilai koalisi B2 tanpa menghitung ulang.

⚠ Metode Shapley yang dihitung dengan operator yang sama dengan operator penghapusan punya **keuntungan kandang**. Karena itu laporkan juga hasil silang (Shapley rata-rata dinilai dengan operator distribusi, dan sebaliknya).

In [ ]:
def load_seed_mean(path, to_group):
    d = pd.read_csv(path, index_col=[0, 1])
    return to_group(d).groupby(level=1).mean().reindex(IDX)

ATT = {}
for k in ("xgb", "rf"):
    ATT[(k, "TreeSHAP (fitur→kelompok)")] = X.group_abs(pd.read_csv(RES / f"xai_shap_{k}.csv", index_col=0), FMAP)[GROUPS].reindex(IDX)
    ATT[(k, "LIME fitur (→kelompok)")] = load_seed_mean(RES / f"xai_lime_{k}.csv", lambda d: X.group_abs(d, FMAP)[GROUPS])
for k in MODELS:
    ATT[(k, "Shapley kelompok (rata-rata)")] = PHI[(k, "rata-rata")].abs()[GROUPS]
    ATT[(k, "Shapley kelompok (distribusi)")] = PHI[(k, "distribusi")].abs()[GROUPS]
    ATT[(k, "LIME kelompok")] = load_seed_mean(RES / f"xai_group_lime_{k}.csv", lambda d: d[GROUPS].abs())
ge_ch = RES / "xai_ge_lstm_channel.csv"
if ge_ch.exists():
    chg = {f"enc:{c}": g for c, g in zip(A["seq_ch"], seq_g)} | {f"dec:{c}": g for c, g in zip(A["tgt_ch"], tgt_g)}
    gm = pd.read_csv(ge_ch, index_col=[0, 1]).groupby(level=1).mean().reindex(IDX)
    ATT[("lstm", "GradientExplainer (|Σφ| kelompok)")] = gm.T.groupby(lambda c: chg[c]).sum().T.abs().reindex(columns=GROUPS, fill_value=0)

n = len(IDX); ar = np.arange(n)
def curves(order, Vm):
    full = Vm[-1]; empty = Vm[0]
    dele, ins = [], []
    for j in range(MG + 1):
        pres = np.ones((n, MG), bool); pres[ar[:, None], order[:, :j]] = False
        dele.append(np.abs(full - Vm[X.coalition_index(pres), ar]))
        pres = np.zeros((n, MG), bool); pres[ar[:, None], order[:, :j]] = True
        ins.append(np.abs(full - Vm[X.coalition_index(pres), ar]))
    return np.array(dele), np.array(ins), np.abs(full - empty)        # [M+1, n] ×2, [n]

def summarize(dele, ins, scale):
    s = scale.mean()
    return {"deletion_AUC": dele[1:MG].mean(), "insertion_AUC": ins[1:MG].mean(),
            "deletion_AUC_norm": dele[1:MG].mean() / s, "insertion_AUC_norm": ins[1:MG].mean() / s}

frows, CURVES = [], {}
rng = np.random.default_rng(SEED)
for k in MODELS:
    for op in ("rata-rata", "distribusi"):
        Vm = V[(k, op)]
        rnd = [curves(rng.permuted(np.tile(np.arange(MG), (n, 1)), axis=1), Vm) for _ in range(N_RAND)]
        rd, ri, sc = np.mean([r[0] for r in rnd], 0), np.mean([r[1] for r in rnd], 0), rnd[0][2]
        base = summarize(rd, ri, sc)
        frows.append({"model": k, "operator_hapus": op, "metode": "acak (pembanding)", **base})
        CURVES[(k, op, "acak (pembanding)")] = rd.mean(1)
        for (kk, meth), att in ATT.items():
            if kk != k: continue
            order = np.argsort(-att[GROUPS].to_numpy(), axis=1, kind="stable")
            d_, i_, sc = curves(order, Vm)
            s = summarize(d_, i_, sc)
            kandang = (meth.startswith("Shapley") and f"({op})" in meth)
            frows.append({"model": k, "operator_hapus": op, "metode": meth, **s,
                          "deletion_vs_acak": s["deletion_AUC"] - base["deletion_AUC"],
                          "insertion_vs_acak": s["insertion_AUC"] - base["insertion_AUC"], "keuntungan_kandang": kandang})
            CURVES[(k, op, meth)] = d_.mean(1)
FA = pd.DataFrame(frows)
for c in FA.columns[3:]:
    if FA[c].dtype != bool and FA[c].dtype != object: FA[c] = FA[c].round(5)
display(FA); FA.to_csv(RES / "post_faithfulness.csv", index=False)
print("Peringkat metode (operator distribusi, deletion AUC besar = lebih setia):")
display(FA[FA["operator_hapus"] == "distribusi"].sort_values(["model", "deletion_AUC"], ascending=[True, False])[["model", "metode", "deletion_AUC_norm", "insertion_AUC_norm", "keuntungan_kandang"]])

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for a, k in zip(axs, MODELS):
    for (kk, op, meth), c in CURVES.items():
        if kk == k and op == "distribusi":
            a.plot(range(MG + 1), c, marker=".", ls="--" if meth.startswith("acak") else "-", label=meth)
    a.set(title=f"Deletion — {k.upper()} (operator distribusi)", xlabel="jumlah kelompok dihapus", ylabel="|f(x) − f(x tanpa kelompok)| (k_c)")
    a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "post_faithfulness.png", dpi=150, bbox_inches="tight"); plt.show()

## B5. Analisis bias — MBE per jam, musim, kondisi langit + dekomposisi galat

MSE = MBE² + Var(galat): porsi MBE² menunjukkan seberapa besar galat berasal dari bias sistematis (bisa dikoreksi) dibanding galat acak.

In [ ]:
BB = {}
for split in ("val", "test"):
    d = P[EVAL[split]].copy()
    keys = {"jam": d["target_hour_wib"], "musim": d["target_day"].dt.month.map(X.SEASON),
            "kc_bin": pd.cut(d["y_kc"], [-np.inf, .4, .8, np.inf], labels=["berawan (<0,4)", "sebagian (0,4–0,8)", "cerah (>0,8)"])}
    err = pd.DataFrame({c.replace("pred_", ""): d[c] - d["y_ghi"] for c in BASE + ML})
    for by, key in keys.items():
        t = err.groupby(key, observed=True).mean(); t["n"] = err.groupby(key, observed=True).size()
        BB[(split, by)] = t; t.round(2).to_csv(RES / f"post_bias_{split}_{by}.csv")
    dec = pd.DataFrame({"MBE": err.mean(), "MBE²": err.mean() ** 2, "Var_galat": err.var(ddof=0), "MSE": (err ** 2).mean()})
    dec["%_MSE_dari_bias"] = 100 * dec["MBE²"] / dec["MSE"]
    BB[(split, "dekomposisi")] = dec; dec.round(3).to_csv(RES / f"post_bias_{split}_dekomposisi.csv")
for by in ("dekomposisi", "kc_bin", "musim", "jam"):
    print(f"Uji 2020 — MBE per {by}:" if by != "dekomposisi" else "Uji 2020 — dekomposisi galat:")
    display(BB[("test", by)].round(1 if by != "dekomposisi" else 2))

fig, axs = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, split, title in zip(axs, ("val", "test"), ("Validasi 2019", "Uji 2020")):
    t = BB[(split, "jam")]
    for c in ["clim", "mos", "rf", "xgb", "lstm"]:
        a.plot(t.index, t[c], marker="o", label=c.upper())
    a.axhline(0, c="k", lw=.6); a.set(title=f"MBE per jam — {title}", xlabel="jam WIB", ylabel="MBE (W/m²)")
axs[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "post_bias_by_hour.png", dpi=150, bbox_inches="tight"); plt.show()

## Ringkasan

In [ ]:
s = summ.xs("RMSE uji 2020", level="besaran")
pr = pairs.set_index("pasangan")
geo = {c.split("|")[1]: int(rank.loc["geometri_waktu", c]) for c in rank if c.startswith("lstm")}
nwp = {c: int(rank.loc["nwp_radiasi", c]) for c in rank if c.endswith("distribusi")}
fd = FA[(FA["operator_hapus"] == "distribusi") & (FA["metode"] != "acak (pembanding)")]
best = fd.loc[fd.groupby("model")["deletion_AUC"].idxmax(), ["model", "metode"]].set_index("model")["metode"].to_dict()
dec = BB[("test", "dekomposisi")]
print(f"""Salin ke decisions.md (semua baris berlabel PASCA-HASIL):
- {pd.Timestamp.today():%Y-%m-%d} | PASCA-HASIL B1 variasi seed ({N_SEEDS} seed, hyperparameter tetap): RMSE uji rata2±std RF {s.loc['rf','rata2']:.1f}±{s.loc['rf','std']:.1f}, XGB {s.loc['xgb','rata2']:.1f}±{s.loc['xgb','std']:.1f}, LSTM {s.loc['lstm','rata2']:.1f}±{s.loc['lstm','std']:.1f}. Selisih LSTM−RF seed 42 {pr.loc['lstm − rf', f'selisih_seed{SEED}']:+.1f}; rentang semua pasangan seed [{pr.loc['lstm − rf','min']:+.1f}; {pr.loc['lstm − rf','maks']:+.1f}], LSTM lebih buruk pada {pr.loc['lstm − rf','%_pasangan_a_lebih_buruk']:.0f}% pasangan. Model utama tidak diganti. | results/post_seed_*.csv | Bab IV (kekokohan)
- {pd.Timestamp.today():%Y-%m-%d} | PASCA-HASIL B2 baseline Shapley (rata-rata vs distribusi {K_BG} hari latih): peringkat geometri LSTM {geo}; nwp_radiasi (distribusi) {nwp}. ρ rata-rata vs distribusi: {dict(zip(agr.query("pembanding == 'rata-rata vs distribusi'")['model'], agr.query("pembanding == 'rata-rata vs distribusi'")['rho']))}. | results/post_shapley_baseline*.csv | Bab IV.4, Batasan
- {pd.Timestamp.today():%Y-%m-%d} | PASCA-HASIL B3 XAI val 2019 vs uji 2020: ρ antar tahun {dict(zip(agr3['model'] + '|' + agr3['metode'], agr3['rho_2019_vs_2020']))}. | results/post_xai_val2019*.csv | Bab IV.4
- {pd.Timestamp.today():%Y-%m-%d} | PASCA-HASIL B4 faithfulness (deletion, operator distribusi): metode AUC terbesar per model {best} (catat keuntungan kandang Shapley-distribusi). | results/post_faithfulness.csv | Bab IV.4
- {pd.Timestamp.today():%Y-%m-%d} | PASCA-HASIL B5 bias uji 2020: MBE RF {dec.loc['rf','MBE']:+.1f}, XGB {dec.loc['xgb','MBE']:+.1f}, LSTM {dec.loc['lstm','MBE']:+.1f} W/m²; porsi MSE dari bias LSTM {dec.loc['lstm','%_MSE_dari_bias']:.1f}%. | results/post_bias_*.csv | Bab IV.2""")
print("\nAngka kunci →", report.write_key_numbers().relative_to(ROOT))